# 04 Audiveris Grades: Where Is the OMR Uncertain?

Audiveris is not an end-to-end neural network, so there are no logits to extract.
Instead every symbol it keeps (an *Inter*, short for "interpretation") carries two
scores, both saved in the `.omr` project file:

```text
grade       intrinsic quality: how well the glyph matches its shape, on its own
ctx-grade   contextual quality: grade after support from related symbols
            (a head supported by a stem, a stem by a beam, ...)
```

Both lie in `[0, 1]`. They are **not calibrated probabilities**. Treat them as a
relative ordering of how sure Audiveris is.

This notebook only reads an existing `.omr` file. It runs nothing and patches
nothing. The question is narrow:

```text
Where on the page is Audiveris confident, and where is it not?
```

In the project's terms this is evidence in `U_image`: each grade is attached to a
bounding box in pixel coordinates. The last section lowers the grades into a
positional field and a per-staff `x` profile, the shape we will later want to put
next to audio evidence.

In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.patches import Rectangle

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["image.cmap"] = "gray"

# Works whether Jupyter was started from the repo root or from notebooks/.
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

# Low grade -> dark orange, high grade -> pale orange: uncertain symbols stand out.
GRADE_CMAP = ListedColormap(plt.cm.Oranges(np.linspace(0.95, 0.25, 256)))
GRADE_CMAP.set_bad((0, 0, 0, 0))
POINT_COLOR = "#2a78d6"
INK_MUTED = "#52514e"

## 1. Open an Audiveris Project

A `.omr` file is a zip archive:

```text
book.xml               which input file, which processing steps ran
sheet#N/sheet#N.xml    everything Audiveris found on page N, including grades
sheet#N/BINARY.png     the binarized page Audiveris worked on
```

`springar.omr` is the richest result so far (about 200 noteheads). The other files in
`omr_output/audiveris/` work too, but some of them detected very little.

In [ ]:
OMR_PATH = REPO / "omr_output" / "audiveris" / "springar.omr"
SHEET = 1

archive = zipfile.ZipFile(OMR_PATH)
book = ET.fromstring(archive.read("book.xml"))
sheet = ET.fromstring(archive.read(f"sheet#{SHEET}/sheet#{SHEET}.xml"))

picture = sheet.find("picture")
page_w, page_h = int(picture.get("width")), int(picture.get("height"))
interline = int(sheet.find("scale/interline").get("main"))


def load_page_image():
    # Prefer the original input. Fall back to Audiveris's own binarized page if the
    # input is a PDF, has moved, or does not match the coordinates in the sheet.
    source = Path(book.get("path"))
    image = cv2.imread(str(source), cv2.IMREAD_GRAYSCALE)
    if image is not None and image.shape == (page_h, page_w):
        return image, source.name
    data = np.frombuffer(archive.read(f"sheet#{SHEET}/BINARY.png"), np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_GRAYSCALE), "BINARY.png from the .omr"


page, page_source = load_page_image()
print(f"page: {page_w} x {page_h} px from {page_source}")
print(f"interline (staff-line spacing): {interline} px")

## 2. Parse Every Inter and Its Grade

Inters live in one *SIG* (symbol interpretation graph) per system. We keep, for
each inter:

- what it is: `kind` (the XML tag) and `shape`
- the two grades
- where it is: bounding `box = (x, y, w, h)` in page pixels, plus `staff` and
  `pitch` when present (`pitch` is a staff step, `0` = middle line, positive = down)

Some inters are **compound**: `head-chord`, `rest-chord`, `beam-group`,
`staff-barline` and the like group other inters, and their box covers their
members. Compound inters have no `shape`. We separate them out so a chord's box
does not hide its own noteheads.

In [ ]:
def parse_inters(sheet):
    rows = []
    for system in sheet.iter("system"):
        for el in system.findall("sig/inters/*"):
            bounds = el.find("bounds")
            if bounds is None:
                continue
            grade = float(el.get("grade", "nan"))
            rows.append(
                dict(
                    id=int(el.get("id")),
                    kind=el.tag,
                    shape=el.get("shape"),
                    grade=grade,
                    ctx_grade=float(el.get("ctx-grade", grade)),
                    staff=int(el.get("staff")) if el.get("staff") else None,
                    pitch=float(el.get("pitch")) if el.get("pitch") else None,
                    system=int(system.get("id")),
                    box=tuple(round(float(bounds.get(k))) for k in "xywh"),
                )
            )
    return rows


inters = parse_inters(sheet)
symbols = [r for r in inters if r["shape"] is not None]
compounds = [r for r in inters if r["shape"] is None]

by_kind = defaultdict(list)
for r in symbols:
    by_kind[r["kind"]].append(r)

print(f"{len(inters)} inters: {len(symbols)} symbols, {len(compounds)} compound\n")
print(f"{'kind':<18}{'n':>5}{'grade':>9}{'ctx':>9}{'min ctx':>9}")
for kind, rows in sorted(by_kind.items(), key=lambda kv: -len(kv[1])):
    g = np.array([r["grade"] for r in rows])
    c = np.array([r["ctx_grade"] for r in rows])
    print(f"{kind:<18}{len(rows):>5}{g.mean():>9.2f}{c.mean():>9.2f}{c.min():>9.2f}")

# Saved grades rarely go near 0, so the colour scale starts just below the weakest
# one. It covers both grade and ctx-grade, so colours stay comparable between them.
lowest = min(min(r["grade"], r["ctx_grade"]) for r in symbols)
GRADE_NORM = Normalize(np.floor(10 * lowest) / 10, 1)
print(f"\ncolour scale: {GRADE_NORM.vmin:.1f} .. 1.0")

## 3. Intrinsic vs Contextual Grade

Each point is one symbol: `x` = grade, `y` = ctx-grade. Points on the diagonal got
no help from context. Points far above it are weak glyphs that the surrounding
structure "rescued".

Keep in mind that this distribution is **truncated**. Audiveris discards
candidates it considers too weak before it saves the project, so we only see the
survivors. It is not a full picture of what the classifier considered.

In [ ]:
MIN_PER_PANEL = 5
kinds = [k for k, rows in sorted(by_kind.items(), key=lambda kv: -len(kv[1])) if len(rows) >= MIN_PER_PANEL][:8]

ncols = 4
nrows = int(np.ceil(len(kinds) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.6 * nrows), sharex=True, sharey=True, squeeze=False)
for ax, kind in zip(axes.flat, kinds):
    rows = by_kind[kind]
    ax.plot([0, 1], [0, 1], color=INK_MUTED, lw=0.8, ls="--", zorder=1)
    ax.scatter(
        [r["grade"] for r in rows], [r["ctx_grade"] for r in rows],
        s=36, color=POINT_COLOR, alpha=0.6, edgecolor="white", linewidth=0.8, zorder=2,
    )
    ax.set_title(f"{kind}  (n={len(rows)})", fontsize=10)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.grid(color="#e6e6e3", lw=0.6)
    ax.set_axisbelow(True)
for ax in axes.flat[len(kinds):]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("grade (intrinsic)")
for ax in axes[:, 0]:
    ax.set_ylabel("ctx-grade")
fig.suptitle("Audiveris grades per symbol kind", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 4. Confidence on the Page

Every symbol box is drawn on a faded page, coloured by its grade. **Dark orange
means low grade (uncertain)** and pale means confident. The colour scale starts at
the weakest saved grade (see the end of section 2), not at 0. Uncertain boxes are drawn
last so they stay on top.

Switch `COLOR_BY` to `"grade"` to see the glyph-only view before context was taken
into account.

In [ ]:
COLOR_BY = "ctx_grade"  # or "grade"


def faded(image, amount=0.55):
    # Lighten ink toward white so coloured boxes read on top of it.
    return (255 - (255 - image.astype(np.float32)) * (1 - amount)).astype(np.uint8)


def draw_grades(ax, rows, value):
    for r in sorted(rows, key=lambda r: -r[value]):
        x, y, w, h = r["box"]
        color = GRADE_CMAP(GRADE_NORM(r[value]))
        ax.add_patch(Rectangle((x, y), w, h, facecolor=(*color[:3], 0.35), edgecolor=color, lw=1.2))


fig, ax = plt.subplots(figsize=(13, 13 * page_h / page_w))
ax.imshow(faded(page), vmin=0, vmax=255)
draw_grades(ax, symbols, COLOR_BY)
ax.set_title(f"{OMR_PATH.name}: symbols coloured by {COLOR_BY}", loc="left")
ax.axis("off")
fig.colorbar(plt.cm.ScalarMappable(GRADE_NORM, GRADE_CMAP), ax=ax, fraction=0.025, pad=0.01, label=COLOR_BY)
plt.tight_layout()
plt.show()

Zoom in on one system to compare the boxes with the ink. Change `SYSTEM` to move
around the page.

In [ ]:
SYSTEM = 1

rows = [r for r in symbols if r["system"] == SYSTEM]
x0 = max(min(r["box"][0] for r in rows) - 2 * interline, 0)
x1 = min(max(r["box"][0] + r["box"][2] for r in rows) + 2 * interline, page_w)
y0 = max(min(r["box"][1] for r in rows) - 2 * interline, 0)
y1 = min(max(r["box"][1] + r["box"][3] for r in rows) + 2 * interline, page_h)

fig, ax = plt.subplots(figsize=(16, 16 * (y1 - y0) / (x1 - x0)))
ax.imshow(faded(page, 0.35), vmin=0, vmax=255)
draw_grades(ax, rows, COLOR_BY)
ax.set_xlim(x0, x1)
ax.set_ylim(y1, y0)
ax.set_title(f"system {SYSTEM}: {len(rows)} symbols coloured by {COLOR_BY}", loc="left")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Look at the Weakest Symbols

A ranked gallery is the quickest way to judge whether a low grade means "wrong",
"right but ugly", or "right but damaged print". The second gallery shows the symbols
that context lifted the most: weak glyphs that Audiveris kept because the
surrounding structure supported them.

In [ ]:
N_GALLERY = 16


def show_gallery(rows, title, ncols=8):
    nrows = int(np.ceil(len(rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2 * ncols, 2.5 * nrows), squeeze=False)
    pad = interline
    for ax, r in zip(axes.flat, rows):
        x, y, w, h = r["box"]
        crop = page[max(y - pad, 0) : y + h + pad, max(x - pad, 0) : x + w + pad]
        ax.imshow(crop, vmin=0, vmax=255)
        ax.add_patch(Rectangle((min(pad, x), min(pad, y)), w, h, fill=False,
                               edgecolor=GRADE_CMAP(GRADE_NORM(r["ctx_grade"])), lw=1.5))
        ax.set_title(f"{r['shape']}\ng {r['grade']:.2f}  ctx {r['ctx_grade']:.2f}", fontsize=7)
        ax.axis("off")
    for ax in axes.flat[len(rows):]:
        ax.axis("off")
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


weakest = sorted(symbols, key=lambda r: r["ctx_grade"])[:N_GALLERY]
show_gallery(weakest, f"{N_GALLERY} lowest ctx-grade symbols")

lifted = sorted(symbols, key=lambda r: r["grade"] - r["ctx_grade"])[:N_GALLERY]
show_gallery(lifted, f"{N_GALLERY} symbols most lifted by context (ctx-grade − grade)")

## 6. Lower the Grades into Positional Fields

So far the grades are attached to a list of symbols. Following `type-lowering.pdf`,
we now lower them onto the page's index space:

```text
F_conf : U_image -> [0, 1] ∪ {undefined}
F_conf(p) = min ctx-grade of all symbols whose box covers pixel p
```

We take the **minimum** so that a pixel is only as trustworthy as the weakest
interpretation claiming it. Pixels covered by no symbol are left undefined
(transparent). They are not "confident", they are just unexplained.

In [ ]:
def confidence_field(rows, value="ctx_grade"):
    field = np.full((page_h, page_w), np.nan, dtype=np.float32)
    for r in rows:
        x, y, w, h = r["box"]
        patch = field[max(y, 0) : y + h, max(x, 0) : x + w]
        np.fmin(patch, r[value], out=patch)  # fmin ignores NaN, so empty pixels take the value
    return field


field = confidence_field(symbols, COLOR_BY)
covered = ~np.isnan(field)
ink = page < 128
print(f"pixels covered by some symbol: {covered.mean():.1%}")
print(f"ink pixels covered by some symbol: {(covered & ink).sum() / ink.sum():.1%}  (staff lines are not symbols)")

fig, ax = plt.subplots(figsize=(13, 13 * page_h / page_w))
ax.imshow(faded(page, 0.7), vmin=0, vmax=255)
im = ax.imshow(field, cmap=GRADE_CMAP, norm=GRADE_NORM, alpha=0.75, interpolation="nearest")
ax.set_title(f"F_conf: min {COLOR_BY} per pixel", loc="left")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label=COLOR_BY)
plt.tight_layout()
plt.show()

### Per-staff confidence along `x`

Inside one staff, `x` is the axis that will eventually be aligned with time in
`U_audio`. Here we project the field onto `x` for each staff, using only the
symbols that mark musical events (noteheads and rests). The result is one 1-D
confidence profile per staff:

```text
P_staff(x) = min ctx-grade of the event symbols on that staff covering column x
```

Gaps are columns with no event symbol at all.

In [ ]:
EVENT_KINDS = {"head", "rest", "grace-chord"}


def staff_extents(sheet):
    staves = []
    for staff in sheet.iter("staff"):
        lines = staff.findall("lines/line")
        if not lines:
            continue
        ys = [float(p.get("y")) for p in lines[0].findall("point") + lines[-1].findall("point")]
        staves.append(dict(id=int(staff.get("id")), left=int(staff.get("left")),
                           right=int(staff.get("right")), top=min(ys), bottom=max(ys)))
    return staves


staves = staff_extents(sheet)
fig, axes = plt.subplots(len(staves), 1, figsize=(14, 1.1 * len(staves)), sharex=True, squeeze=False)
for ax, staff in zip(axes[:, 0], staves):
    profile = np.full(page_w, np.nan)
    events = [r for r in symbols if r["staff"] == staff["id"] and r["kind"] in EVENT_KINDS]
    for r in events:
        x, _, w, _ = r["box"]
        np.fmin(profile[x : x + w], r[COLOR_BY], out=profile[x : x + w])
    ax.fill_between(np.arange(page_w), profile, color=POINT_COLOR, alpha=0.35, lw=0, step="mid")
    ax.plot(np.arange(page_w), profile, color=POINT_COLOR, lw=1, drawstyle="steps-mid")
    ax.axvspan(0, staff["left"], color="#f0f0ed", lw=0)
    ax.axvspan(staff["right"], page_w, color="#f0f0ed", lw=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.set_ylabel(f"staff {staff['id']}", rotation=0, ha="right", va="center", fontsize=8)
    ax.text(page_w, 0.5, f" {len(events)} events", va="center", fontsize=7, color=INK_MUTED)
axes[-1, 0].set_xlim(0, page_w)
axes[-1, 0].set_xlabel("x (page pixels)")
fig.suptitle(f"min {COLOR_BY} of noteheads and rests along each staff", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 7. What to Look For

- Are the dark boxes actually wrong, or just unusual glyphs (grace notes, ornaments,
  damaged print)? The galleries answer this faster than the overview.
- Do low-confidence regions cluster, for example at the start of a system (clef/key),
  in dense beamed passages, or near slurs and ornaments?
- Which symbols are rescued by context? These are interpretations that are only as
  good as the structure around them.
- What ink is *not* covered at all (section 6)? Uncovered ink that is not a staff line
  is something Audiveris saw but did not interpret.

Possible next steps:

```text
homr segmentation:   per-pixel class probabilities  -> a second F_conf on U_image
homr transformer:    per-token entropy + attention  -> per-staff confidence along x
compare:             where do Audiveris and homr disagree or are both unsure?
```